# HR Analytics & Employee Attrition Analysis

## 03 - Business Insights

This notebook summarizes the key findings identified during
exploratory data analysis.

### Objectives
- Quantify the main attrition patterns
- Compare employee groups
- Identify areas that warrant further investigation
- Create concise metrics for reporting and visualization
- Prepare findings for the Power BI dashboard and project report

In [ ]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/cleaned/HR_Python_Cleaned_Data.csv")

## 1. Overall Workforce KPIs

In [3]:
total_employees = len(df)

employees_left = (df["Attrition"] == "Yes").sum()

employees_stayed = (df["Attrition"] == "No").sum()

attrition_rate = employees_left / total_employees * 100

average_age = df["Age"].mean()

average_income = df["MonthlyIncome"].mean()

average_years_at_company = df["YearsAtCompany"].mean()

In [4]:
overall_kpis = pd.DataFrame({
    "Metric": [
        "Total Employees",
        "Employees Who Left",
        "Employees Who Stayed",
        "Attrition Rate",
        "Average Age",
        "Average Monthly Income",
        "Average Years at Company"
    ],
    "Value": [
        total_employees,
        employees_left,
        employees_stayed,
        round(attrition_rate, 2),
        round(average_age, 2),
        round(average_income, 2),
        round(average_years_at_company, 2)
    ]
})

overall_kpis

,Metric,Value
0,Total Employees,1470.00
1,Employees Who Left,237.00
2,Employees Who Stayed,1233.00
3,Attrition Rate,16.12
4,Average Age,36.92
5,Average Monthly Income,6502.93
6,Average Years at Company,7.01


## 2. Attrition by Department

In [5]:
department_insights = (
    df.groupby("Department")
      .agg(
          Total_Employees=("EmployeeNumber", "count"),
          Employees_Left=("Attrition", lambda x: (x == "Yes").sum())
      )
)

department_insights["Attrition_Rate"] = (
    department_insights["Employees_Left"]
    / department_insights["Total_Employees"] * 100
)

department_insights = department_insights.round(2)

department_insights.sort_values(
    "Attrition_Rate",
    ascending=False
)

,Total_Employees,Employees_Left,Attrition_Rate
Department,,,
Sales,446,92,20.63
Human Resources,63,12,19.05
Research & Development,961,133,13.84


## 3. Attrition by Job Role

In [6]:
jobrole_insights = (
    df.groupby("JobRole")
      .agg(
          Total_Employees=("EmployeeNumber", "count"),
          Employees_Left=("Attrition", lambda x: (x == "Yes").sum())
      )
)

jobrole_insights["Attrition_Rate"] = (
    jobrole_insights["Employees_Left"]
    / jobrole_insights["Total_Employees"] * 100
)

jobrole_insights = jobrole_insights.round(2)

jobrole_insights.sort_values(
    "Attrition_Rate",
    ascending=False
)

,Total_Employees,Employees_Left,Attrition_Rate
JobRole,,,
Sales Representative,83,33,39.76
Laboratory Technician,259,62,23.94
Human Resources,52,12,23.08
Sales Executive,326,57,17.48
Research Scientist,292,47,16.10
Manufacturing Director,145,10,6.90
Healthcare Representative,131,9,6.87
Manager,102,5,4.90
Research Director,80,2,2.50


## 4. Attrition by Overtime

In [8]:
overtime_insights = (
    df.groupby("OverTime")
      .agg(
          Total_Employees=("EmployeeNumber", "count"),
          Employees_Left=("Attrition", lambda x: (x == "Yes").sum())
      )
)

overtime_insights["Attrition_Rate"] = (
    overtime_insights["Employees_Left"]
    / overtime_insights["Total_Employees"] * 100
)

overtime_insights.round(2)

,Total_Employees,Employees_Left,Attrition_Rate
OverTime,,,
No,1054,110,10.44
Yes,416,127,30.53


In [9]:
overtime_rate_difference = (
    overtime_insights.loc["Yes", "Attrition_Rate"]
    - overtime_insights.loc["No", "Attrition_Rate"]
)

print(f"Attrition-rate difference: {overtime_rate_difference:.2f} percentage points")

Attrition-rate difference: 20.09 percentage points


## 5. Attrition by Tenure

In [10]:
tenure_insights = (
    df.groupby("Tenure_Group")
      .agg(
          Total_Employees=("EmployeeNumber", "count"),
          Employees_Left=("Attrition", lambda x: (x == "Yes").sum())
      )
)

tenure_insights["Attrition_Rate"] = (
    tenure_insights["Employees_Left"]
    / tenure_insights["Total_Employees"] * 100
)

tenure_insights.round(2)

,Total_Employees,Employees_Left,Attrition_Rate
Tenure_Group,,,
0-1 Years,215,75,34.88
10+ Years,246,20,8.13
2-5 Years,561,87,15.51
6-10 Years,448,55,12.28


In [11]:
tenure_difference = (
    tenure_insights.loc["0-1 Years", "Attrition_Rate"]
    - tenure_insights.loc["10+ Years", "Attrition_Rate"]
)

print(f"Difference: {tenure_difference:.2f} percentage points")

Difference: 26.75 percentage points


## 6. Attrition by Age Group

In [12]:
age_insights = (
    df.groupby("Age_Group")
      .agg(
          Total_Employees=("EmployeeNumber", "count"),
          Employees_Left=("Attrition", lambda x: (x == "Yes").sum())
      )
)

age_insights["Attrition_Rate"] = (
    age_insights["Employees_Left"]
    / age_insights["Total_Employees"] * 100
)

age_insights.round(2)

,Total_Employees,Employees_Left,Attrition_Rate
Age_Group,,,
25-34,554,112,20.22
35-44,505,51,10.10
45-54,245,25,10.20
55+,69,11,15.94
Under 25,97,38,39.18


## 7. Income and Attrition

In [13]:
income_by_attrition = (
    df.groupby("Attrition")["MonthlyIncome"]
      .agg(["count", "mean", "median", "min", "max"])
      .round(2)
)

income_by_attrition

,count,mean,median,min,max
Attrition,,,,,
No,1233,6832.74,5204.0,1051,19999
Yes,237,4787.09,3202.0,1009,19859


In [14]:
income_difference = (
    df.loc[df["Attrition"] == "No", "MonthlyIncome"].mean()
    - df.loc[df["Attrition"] == "Yes", "MonthlyIncome"].mean()
)

print(f"Average income difference: {income_difference:.2f}")

Average income difference: 2045.65


## 8. Satisfaction and Work-Life Balance

In [15]:
satisfaction_summary = (
    df.groupby("Attrition")
      .agg(
          Average_Job_Satisfaction=("JobSatisfaction", "mean"),
          Average_Environment_Satisfaction=("EnvironmentSatisfaction", "mean"),
          Average_Work_Life_Balance=("WorkLifeBalance", "mean"),
          Average_Job_Involvement=("JobInvolvement", "mean"),
          Average_Relationship_Satisfaction=("RelationshipSatisfaction", "mean")
      )
      .round(2)
)

satisfaction_summary

,Average_Job_Satisfaction,Average_Environment_Satisfaction,Average_Work_Life_Balance,Average_Job_Involvement,Average_Relationship_Satisfaction
Attrition,,,,,
No,2.78,2.77,2.78,2.77,2.73
Yes,2.47,2.46,2.66,2.52,2.60


## 9. Key Attrition Findings

In [16]:
key_findings = pd.DataFrame({
    "Analysis": [
        "Overall Attrition",
        "Overtime",
        "0-1 Years Tenure",
        "10+ Years Tenure",
        "Under 25 Age Group"
    ],
    "Metric": [
        "Overall Attrition Rate",
        "Attrition Rate",
        "Attrition Rate",
        "Attrition Rate",
        "Attrition Rate"
    ],
    "Value": [
        f"{attrition_rate:.2f}%",
        f"{overtime_insights.loc['Yes', 'Attrition_Rate']:.2f}%",
        f"{tenure_insights.loc['0-1 Years', 'Attrition_Rate']:.2f}%",
        f"{tenure_insights.loc['10+ Years', 'Attrition_Rate']:.2f}%",
        f"{age_insights.loc['Under 25', 'Attrition_Rate']:.2f}%"
    ]
})

key_findings

,Analysis,Metric,Value
0,Overall Attrition,Overall Attrition Rate,16.12%
1,Overtime,Attrition Rate,30.53%
2,0-1 Years Tenure,Attrition Rate,34.88%
3,10+ Years Tenure,Attrition Rate,8.13%
4,Under 25 Age Group,Attrition Rate,39.18%


## 10. Summary

### Overall Workforce
- The dataset contains 1,470 employees.
- 237 employees are recorded as having left.
- 1,233 employees are recorded as having stayed.
- The overall attrition rate is 16.12%.

### Key Observed Patterns
- Attrition rates differ across departments and job roles.
- Employees working overtime show a higher observed attrition rate than employees not working overtime.
- Employees in the 0-1 year tenure group show a higher observed attrition rate than longer-tenured groups.
- The Under 25 age group has the highest observed attrition rate among the defined age groups.
- Income, satisfaction, work-life balance, and other employee characteristics should be considered together when interpreting attrition patterns.

### Important Interpretation
These findings describe associations within this dataset. They do not establish that any individual characteristic directly causes employee attrition.